# Step-by-Step MCP (Model Context Protocol) Experiment

This notebook walks you through building a simple **MCP server** from scratch — the same way you explored RAG step-by-step.

---

## What is MCP?

**Model Context Protocol (MCP)** is an open standard (created by Anthropic) that defines how AI models talk to external tools, data sources, and services.

Think of it like a **USB standard for AI**: instead of every AI app inventing its own way to call a calculator, read a file, or query a database, MCP is the universal plug.

```
  ┌─────────────────────────────────┐
  │         MCP Client              │  ← e.g. Claude Desktop, your LLM app
  │  (asks: "what tools exist?")    │
  └───────────────┬─────────────────┘
                  │  JSON-RPC over stdio / HTTP
  ┌───────────────▼─────────────────┐
  │         MCP Server              │  ← what WE are building
  │  (exposes: tools, resources,    │
  │            prompts)             │
  └─────────────────────────────────┘
```

### Key concepts

| Concept | What it is | Example |
|---|---|---|
| **Tool** | A function the LLM can call | `add(a, b)`, `get_weather(city)` |
| **Resource** | A readable data source | A file, a DB row, a URL |
| **Prompt** | A reusable prompt template | `summarise_text(text)` |

In this notebook we focus on **Tools**, which are the most common use-case.

## Step 1 — Install the MCP SDK

The official Python SDK is `mcp`. It ships a `FastMCP` helper class that removes all the boilerplate.

In [ ]:
# Install the MCP Python SDK
# 'mcp[cli]' includes the CLI tools (mcp dev, mcp run) on top of the core library
%pip install -r ../requirements.txt

## Step 2 — Understand the anatomy of an MCP server

A minimal server has three parts:

1. **Create a `FastMCP` instance** — this is your server object.
2. **Decorate functions with `@mcp.tool()`** — each decorated function becomes a callable tool that any MCP client can discover and invoke.
3. **Start the server** — `mcp.run()` starts the event loop and listens for client messages over **stdio** (the simplest transport).

```python
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("my-server")        # 1. create server

@mcp.tool()                        # 2. register a tool
def greet(name: str) -> str:
    """Return a greeting for the given name."""
    return f"Hello, {name}!"

mcp.run()                          # 3. start the server
```

> **Why stdio?** It's the simplest transport: the client spawns the server as a subprocess and communicates via stdin/stdout. No network config needed.

## Step 3 — Write your first MCP server

We'll write a small **calculator server** that exposes four arithmetic tools.

Because MCP servers are long-running processes (not notebook code), we write the server to a `.py` file and run it from the terminal.

In [1]:
# Write the server code to disk so we can run it as a subprocess
server_code = '''
# calculator_server.py  —  a simple MCP server exposing 4 arithmetic tools

from mcp.server.fastmcp import FastMCP

# Create the MCP server instance.
# The name ("Calculator") is what MCP clients see when they list available servers.
mcp = FastMCP("Calculator")


@mcp.tool()
def add(a: float, b: float) -> float:
    """Add two numbers and return the result."""
    return a + b


@mcp.tool()
def subtract(a: float, b: float) -> float:
    """Subtract b from a and return the result."""
    return a - b


@mcp.tool()
def multiply(a: float, b: float) -> float:
    """Multiply two numbers and return the result."""
    return a * b


@mcp.tool()
def divide(a: float, b: float) -> float:
    """Divide a by b and return the result. Raises ValueError if b is zero."""
    if b == 0:
        raise ValueError("Cannot divide by zero")
    return a / b


# mcp.run() starts the server and blocks.
# transport="stdio" means the client communicates via stdin/stdout.
if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

with open("calculator_server.py", "w") as f:
    f.write(server_code.strip())

print("calculator_server.py written!")

calculator_server.py written!


## Step 4 — Inspect the server's tools (without an LLM)

Before wiring up an LLM, let's directly talk to the MCP server using the Python SDK client.

This simulates exactly what Claude Desktop or any MCP client does:
1. Spawn the server as a subprocess.
2. Call `list_tools()` to discover what tools it exposes.
3. Call `call_tool()` to invoke one.

In [2]:
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# Tell the MCP client how to start our server as a child process
server_params = StdioServerParameters(
    command="python",          # interpreter to use
    args=["calculator_server.py"],  # script to run
)

async def explore_server():
    # stdio_client() spawns the server subprocess and opens stdin/stdout pipes
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            # Perform the MCP handshake (version negotiation)
            await session.initialize()

            # ── List all available tools ──────────────────────────────────────
            tools_response = await session.list_tools()
            print("=== Available tools ===")
            for tool in tools_response.tools:
                print(f"  • {tool.name}: {tool.description}")
                print(f"    Input schema: {tool.inputSchema}")

            print()

            # ── Call the 'add' tool ───────────────────────────────────────────
            result = await session.call_tool("add", arguments={"a": 12, "b": 30})
            print(f"=== add(12, 30) ===")
            print(f"  Result: {result.content[0].text}")

            # ── Call the 'divide' tool ────────────────────────────────────────
            result = await session.call_tool("divide", arguments={"a": 100, "b": 4})
            print(f"\n=== divide(100, 4) ===")
            print(f"  Result: {result.content[0].text}")

# Run the async function (works in Jupyter with nest_asyncio)
import nest_asyncio
nest_asyncio.apply()
asyncio.get_event_loop().run_until_complete(explore_server())

=== Available tools ===
  • add: Add two numbers and return the result.
    Input schema: {'properties': {'a': {'title': 'A', 'type': 'number'}, 'b': {'title': 'B', 'type': 'number'}}, 'required': ['a', 'b'], 'title': 'addArguments', 'type': 'object'}
  • subtract: Subtract b from a and return the result.
    Input schema: {'properties': {'a': {'title': 'A', 'type': 'number'}, 'b': {'title': 'B', 'type': 'number'}}, 'required': ['a', 'b'], 'title': 'subtractArguments', 'type': 'object'}
  • multiply: Multiply two numbers and return the result.
    Input schema: {'properties': {'a': {'title': 'A', 'type': 'number'}, 'b': {'title': 'B', 'type': 'number'}}, 'required': ['a', 'b'], 'title': 'multiplyArguments', 'type': 'object'}
  • divide: Divide a by b and return the result. Raises ValueError if b is zero.
    Input schema: {'properties': {'a': {'title': 'A', 'type': 'number'}, 'b': {'title': 'B', 'type': 'number'}}, 'required': ['a', 'b'], 'title': 'divideArguments', 'type': 'object'}



## Step 5 — What just happened?

```
Notebook (MCP Client)
    │
    │  1. spawns subprocess
    ▼
calculator_server.py  (MCP Server)
    │
    │  2. MCP handshake  →  initialize()
    │  3. discover tools →  list_tools()
    │  4. call tool      →  call_tool("add", {"a": 12, "b": 30})
    │  5. returns result ←  {"content": [{"text": "42.0"}]}
    ▼
Notebook prints the answer
```

All communication is **JSON-RPC 2.0 over stdio** — just structured messages exchanged
over the subprocess's stdin and stdout pipe.

The key protocol messages are:

| Message | Direction | What it does |
|---|---|---|
| `initialize` | client → server | Agree on protocol version |
| `tools/list` | client → server | List all registered tools |
| `tools/call` | client → server | Invoke a tool with arguments |
| `result` | server → client | Return value or error |

## Step 6 — Handle errors gracefully

Try calling `divide` with `b=0` and observe how the server's `ValueError` is
surfaced back to the client.

In [3]:
async def test_error_handling():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            try:
                result = await session.call_tool("divide", arguments={"a": 5, "b": 0})
                # If the server returns an error flag, content[0] will describe it
                print(f"Result: {result.content}")
                print(f"Is error: {result.isError}")
            except Exception as e:
                print(f"Client-side exception: {type(e).__name__}: {e}")

asyncio.get_event_loop().run_until_complete(test_error_handling())

Result: [TextContent(type='text', text='Error executing tool divide: Cannot divide by zero', annotations=None, meta=None)]
Is error: True


## Step 7 — Add a Resource to your server

**Resources** are read-only data the LLM can access — like files or database rows.
Expose one with `@mcp.resource("uri-scheme://path")`.

Let's add a resource that returns a math cheat-sheet the LLM can read.

In [4]:
server_code_v2 = '''
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Calculator")

# ── Tools ─────────────────────────────────────────────────────────────────────

@mcp.tool()
def add(a: float, b: float) -> float:
    """Add two numbers."""
    return a + b

@mcp.tool()
def subtract(a: float, b: float) -> float:
    """Subtract b from a."""
    return a - b

@mcp.tool()
def multiply(a: float, b: float) -> float:
    """Multiply two numbers."""
    return a * b

@mcp.tool()
def divide(a: float, b: float) -> float:
    """Divide a by b. Raises ValueError if b is zero."""
    if b == 0:
        raise ValueError("Cannot divide by zero")
    return a / b

# ── Resource ──────────────────────────────────────────────────────────────────
# Resources are identified by a URI. The client can read this with read_resource().

@mcp.resource("cheatsheet://math")
def math_cheatsheet() -> str:
    """A short math reference for the LLM to consult."""
    return """
    Math Cheat-Sheet
    ================
    • Addition       : a + b
    • Subtraction    : a - b
    • Multiplication : a * b
    • Division       : a / b  (b ≠ 0)
    • Power          : a ** b
    • Square root    : a ** 0.5
    • Modulo         : a % b
    """


if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

with open("calculator_server.py", "w") as f:
    f.write(server_code_v2.strip())

print("calculator_server.py updated with a Resource!")

calculator_server.py updated with a Resource!


In [5]:
async def explore_resource():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # List resources — like list_tools() but for readable data
            resources = await session.list_resources()
            print("=== Available resources ===")
            for r in resources.resources:
                print(f"  • {r.uri}  ({r.name})")

            # Read the cheatsheet resource by its URI
            content = await session.read_resource("cheatsheet://math")
            print("\n=== cheatsheet://math ===")
            print(content.contents[0].text)

asyncio.get_event_loop().run_until_complete(explore_resource())

=== Available resources ===
  • cheatsheet://math  (math_cheatsheet)

=== cheatsheet://math ===

    Math Cheat-Sheet
    • Addition       : a + b
    • Subtraction    : a - b
    • Multiplication : a * b
    • Division       : a / b  (b ≠ 0)
    • Power          : a ** b
    • Square root    : a ** 0.5
    • Modulo         : a % b
    


## Step 8 — Wire up Ollama as the MCP Client

Now let's do something powerful: connect your **local Ollama LLM** to the MCP server
so the model can actually decide *which tool to call* and *with what arguments*.

### How tool calling works

```
User: "What is 123 multiplied by 456?"
        │
        ▼
   LLM (Ollama)  ←── tool schemas from MCP server
        │
        │  decides to call: multiply(a=123, b=456)
        ▼
   MCP Server   ──── returns: 56088.0
        │
        ▼
   LLM formats the final answer for the user
```

> **Note:** This requires a model that supports tool/function calling.
> `llama3.2` supports it. Make sure Ollama is running (`ollama serve`).

In [6]:
import json
import ollama

async def llm_with_mcp(user_question: str):
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # ── Step A: get tool schemas from MCP server ──────────────────────
            tools_response = await session.list_tools()

            # Convert MCP tool schemas → Ollama tool format
            # Ollama expects: [{"type": "function", "function": {"name": ..., "description": ..., "parameters": ...}}]
            ollama_tools = [
                {
                    "type": "function",
                    "function": {
                        "name": t.name,
                        "description": t.description,
                        "parameters": t.inputSchema,
                    },
                }
                for t in tools_response.tools
            ]

            # ── Step B: ask Ollama, passing the tool list ─────────────────────
            messages = [{"role": "user", "content": user_question}]
            response = ollama.chat(
                model="llama3.2",
                messages=messages,
                tools=ollama_tools,
            )

            print(f"Question : {user_question}")

            # ── Step C: check if the model wants to call a tool ───────────────
            assistant_msg = response["message"]
            if assistant_msg.get("tool_calls"):
                for tool_call in assistant_msg["tool_calls"]:
                    fn = tool_call["function"]
                    tool_name = fn["name"]
                    tool_args = fn["arguments"]
                    print(f"LLM calls : {tool_name}({tool_args})")

                    # ── Step D: execute the tool via the MCP server ───────────
                    mcp_result = await session.call_tool(tool_name, arguments=tool_args)
                    tool_result = mcp_result.content[0].text
                    print(f"MCP result: {tool_result}")

                    # ── Step E: feed result back to the LLM for a final answer ─
                    messages.append(assistant_msg)   # add the LLM's tool-call message
                    messages.append({
                        "role": "tool",
                        "content": tool_result,
                    })
                    final = ollama.chat(model="llama3.2", messages=messages)
                    print(f"Answer    : {final['message']['content']}")
            else:
                # Model answered directly without using any tool
                print(f"Answer    : {assistant_msg['content']}")

asyncio.get_event_loop().run_until_complete(llm_with_mcp("What is 123 multiplied by 456?"))

Question : What is 123 multiplied by 456?
LLM calls : multiply({'a': 123, 'b': 456})
MCP result: 56088.0
Answer    : The result of multiplying 123 by 456 is 56,088.


In [7]:
# Try more questions!
asyncio.get_event_loop().run_until_complete(llm_with_mcp("Divide 1000 by 8"))

Question : Divide 1000 by 8
LLM calls : divide({'a': 1000, 'b': 8})
MCP result: 125.0
Answer    : The result of dividing 1000 by 8 is 125.


In [8]:
asyncio.get_event_loop().run_until_complete(llm_with_mcp("What is 99 minus 37?"))

Question : What is 99 minus 37?
LLM calls : subtract({'a': '99', 'b': '37'})
MCP result: 62.0
Answer    : The result of subtracting 37 from 99 is 62.


## Step 9 — Add a stateful tool (in-memory note store)

Tools can hold **state** between calls — for example an in-memory store.
This is a common pattern for building memory or session tools.

Below we extend the server with two new tools: `save_note` and `get_notes`.

In [9]:
server_code_v3 = '''
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Calculator")

# ── In-memory state ───────────────────────────────────────────────────────────
# This dict persists for the lifetime of the server process.
_notes: dict[str, str] = {}

# ── Arithmetic tools ──────────────────────────────────────────────────────────

@mcp.tool()
def add(a: float, b: float) -> float:
    """Add two numbers."""
    return a + b

@mcp.tool()
def subtract(a: float, b: float) -> float:
    """Subtract b from a."""
    return a - b

@mcp.tool()
def multiply(a: float, b: float) -> float:
    """Multiply two numbers."""
    return a * b

@mcp.tool()
def divide(a: float, b: float) -> float:
    """Divide a by b. Raises ValueError if b is zero."""
    if b == 0:
        raise ValueError("Cannot divide by zero")
    return a / b

# ── Note tools ────────────────────────────────────────────────────────────────

@mcp.tool()
def save_note(key: str, value: str) -> str:
    """Save a note under the given key. Returns a confirmation message."""
    _notes[key] = value
    return f"Note saved under key \'{key}\'"

@mcp.tool()
def get_notes() -> dict:
    """Return all saved notes as a dictionary."""
    return _notes

# ── Resource ──────────────────────────────────────────────────────────────────

@mcp.resource("cheatsheet://math")
def math_cheatsheet() -> str:
    """A short math reference for the LLM to consult."""
    return """
    Math Cheat-Sheet
    ================
    • Addition       : a + b
    • Subtraction    : a - b
    • Multiplication : a * b
    • Division       : a / b  (b ≠ 0)
    • Square root    : a ** 0.5
    """

if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

with open("calculator_server.py", "w") as f:
    f.write(server_code_v3.strip())

print("calculator_server.py updated with note tools!")

calculator_server.py updated with note tools!


In [10]:
# Test the note tools directly (without Ollama)
async def test_notes():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # Save some notes
            r1 = await session.call_tool("save_note", {"key": "pi", "value": "3.14159"})
            print(r1.content[0].text)

            r2 = await session.call_tool("save_note", {"key": "e", "value": "2.71828"})
            print(r2.content[0].text)

            # Retrieve all notes
            r3 = await session.call_tool("get_notes", {})
            print(f"All notes: {r3.content[0].text}")

asyncio.get_event_loop().run_until_complete(test_notes())

Note saved under key 'pi'
Note saved under key 'e'
All notes: {
  "pi": "3.14159",
  "e": "2.71828"
}


## Step 10 — Use the MCP Dev Inspector (CLI)

The MCP SDK ships a browser-based **Inspector** that lets you visually explore and
test any MCP server — without writing any client code.  
You will need to install nodejs and npm to run this (if in Ubuntu, check the script install_node.sh)

Run this in a **terminal** (not in the notebook):

```bash
# Start the inspector, pointing it at your server
mcp dev calculator_server.py
```

Then open `http://localhost:5173` in your browser.
You'll see:
- All registered **tools** with their schemas
- All **resources** with their URIs
- A **test console** to call tools interactively

---

## Summary — What you built

| Step | What you did |
|---|---|
| 1 | Installed the `mcp` Python SDK |
| 2 | Learned the anatomy of a `FastMCP` server |
| 3 | Wrote a calculator server with 4 arithmetic tools |
| 4 | Used the MCP Python client to discover and call tools |
| 5 | Understood the JSON-RPC protocol messages |
| 6 | Handled tool errors gracefully |
| 7 | Added a Resource (read-only data) to the server |
| 8 | Wired Ollama to the MCP server for LLM-driven tool calls |
| 9 | Added stateful tools (in-memory note store) |
| 10 | Explored the MCP Inspector CLI |

### Where to go next

- **HTTP transport**: replace `stdio` with `streamable-http` to expose your server over the network
- **Real tools**: wrap a database query, a REST API call, or a file system reader
- **Claude Desktop integration**: add your server to `claude_desktop_config.json` and use it from Claude
- **MCP docs**: https://modelcontextprotocol.io/docs